# AI Job Displacement Analysis

LLM-Based Assessment System

# API Key

Google Gemini: https://ai.google.dev/

# Imports

In [28]:
import os
import json
import asyncio
import tempfile
import aiofiles
import pandas as pd
from pathlib import Path
from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Dict, List, Any

# Response Schema

In [14]:
class DimensionScore(BaseModel):
    reasoning: str
    score: int

class TaskAssessment(BaseModel):
    task_predictability: DimensionScore
    interaction_medium: DimensionScore
    social_requirement: DimensionScore
    environmental_stability: DimensionScore
    consequence_of_failure: DimensionScore
    regulatory_barrier: DimensionScore
    economic_arbitrage: DimensionScore

# Gemini Automation Pipeline Class

## Load Occupation Data

In [15]:
def load_job_data(data_path: str) -> List[Dict[str, Any]]:
    """
    Read the prepared JSON dataset of occupations and tasks.
    """
    with open(data_path, "r", encoding="utf-8") as f:
        return json.load(f)

## Checkpoints 
resume support

In [16]:
def load_checkpoint(checkpoint_path: str | Path) -> Dict[str, Dict[str, Any]]:
    """
    Load previously completed task results.
    Returns a dict keyed by task_id for fast lookup.
    """
    path = Path(checkpoint_path)
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return {str(item["task_id"]): item for item in data}
    return {}

async def save_checkpoint(checkpoint_path: str | Path, results: List[Dict[str, Any]]) -> None:
    """
    Atomically write the full list of completed task results.
    """
    async with aiofiles.open(checkpoint_path, "w", encoding="utf-8") as f:
        await f.write(json.dumps(results, indent=2, ensure_ascii=False))

## Prompt

In [17]:
def build_assessment_prompt(task_description: str, job_context: str) -> str:
    """Construct the exact 7-dimension evaluation prompt."""
    return f"""Analyze the following job task and evaluate its automation potential across 7 distinct dimensions.
For each dimension, assign a score from 1 to 5, where:
1 = Very difficult to automate (Strong human advantage / External barrier)
5 = Highly automatable (Strong AI/Machine advantage / No barrier)

Job Context: "{job_context}"
Task to Analyze: "{task_description}"

Dimensions to evaluate (Score 1-5):
1. Task Predictability: Is the internal logic heuristic/intuition-based (1) or strictly algorithmic and rule-based (5)?
2. Interaction Medium: Does the task require complex physical manipulation (1) or is it purely abstract digital processing (5)?
3. Social Requirement: Does the task demand deep empathy and trust-building (1) or can it be executed in total isolation (5)?
4. Environmental Stability: Does the work occur in a highly chaotic/unpredictable environment (1) or a perfectly engineered static environment (5)?
5. Consequence of Failure: Are the stakes of a mistake catastrophic (1) or trivial and easily reversed (5)?
6. Regulatory Barrier: Does the law strictly mandate a certified human (1) or is the output completely unregulated (5)?
7. Economic Arbitrage: Is human labor so cheap that automation ROI is negligible (1) or is human labor highly expensive making ROI massive (5)?

Provide your assessment STRICTLY in the following JSON format. Be specific and justify each score in the reasoning field BEFORE providing the score to ensure sound logic.

{{
  "task_predictability": {{"reasoning": "...", "score": X}},
  "interaction_medium": {{"reasoning": "...", "score": X}},
  "social_requirement": {{"reasoning": "...", "score": X}},
  "environmental_stability": {{"reasoning": "...", "score": X}},
  "consequence_of_failure": {{"reasoning": "...", "score": X}},
  "regulatory_barrier": {{"reasoning": "...", "score": X}},
  "economic_arbitrage": {{"reasoning": "...", "score": X}}
}}"""

## LLM Call 
with retries + semaphore

In [30]:
async def assess_task_automation(
    client: genai.Client,
    semaphore: asyncio.Semaphore,
    task_description: str,
    job_context: str,
    max_retries: int = 4,
    base_backoff: float = 8.0,
) -> Dict[str, Any]:
    """
    Call Gemini for one task.
    Uses a semaphore for concurrency control and exponential back-off on rate limits.
    """
    prompt = build_assessment_prompt(task_description, job_context)

    for attempt in range(max_retries):
        try:
            async with semaphore:
                response = await client.aio.models.generate_content(
                    model="gemini-3.6-flash",
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0.2,
                        response_mime_type="application/json",
                        response_schema=TaskAssessment,
                        system_instruction="You are an expert in labor economics and AI automation.",
                    ),
                )
            return json.loads(response.text)

        except Exception as e:
            error_str = str(e).lower()
            is_rate_limit = any(
                x in error_str for x in ("429", "resource_exhausted", "rate limit", "quota")
            )
            is_transient = is_rate_limit or "timeout" in error_str or "connection" in error_str

            if is_transient and attempt < max_retries - 1:
                wait = base_backoff * (2 ** attempt)
                print(f"  [retry {attempt+1}/{max_retries}] {type(e).__name__}: waiting {wait:.1f}s …")
                await asyncio.sleep(wait)
            else:
                print(f"  Failed after {attempt+1} attempts: {e}")
                return {}

    return {}

## Score Calculation

In [19]:
def calculate_automation_score(
    dimension_scores: Dict[str, Any],
    weights: Dict[str, float],
) -> float:
    """
    Weighted average of the seven dimension scores → 0-100 scale.
    """
    overall = 0.0
    for dim, details in dimension_scores.items():
        if dim in weights and isinstance(details, dict) and "score" in details:
            overall += (details["score"] / 5.0) * 100 * weights[dim]
    return overall

## Evaluate one task 
skips if already done

In [35]:
async def evaluate_single_task(
    client: genai.Client,
    semaphore: asyncio.Semaphore,
    completed: Dict[str, Dict[str, Any]],
    job_title: str,
    job_context: str,
    task: Dict[str, Any],
    weights: Dict[str, float],
) -> Dict[str, Any] | None:
    """
    Evaluate a single task (or return the cached result if it already exists).
    """
    task_id = str(task["task_id"])

    # Resume support
    if task_id in completed:
        return completed[task_id]

    task_desc = task["task_description"]
    importance = float(task.get("importance") or 1.0)

    assessment = await assess_task_automation(
        client, semaphore, task_desc, job_context
    )
    if not assessment:
        return None

    task_score = calculate_automation_score(assessment, weights)

    return {
        "job_title": job_title,
        "task_id": task["task_id"],
        "task_description": task_desc,
        "dimension_assessments": assessment,
        "task_automation_score": round(task_score, 2),
        "importance": importance,
    }

## Process Every Occupation 
orchestrator

In [21]:
async def process_all_occupations(
    client: genai.Client,
    job_data: List[Dict[str, Any]],
    weights: Dict[str, float],
    checkpoint_path: str | Path,
    max_concurrent: int = 8,
) -> List[Dict[str, Any]]:
    """
    Main driver.
    - Loads any existing checkpoint
    - Runs tasks concurrently under a semaphore
    - Checkpoints after every occupation
    """
    completed = load_checkpoint(checkpoint_path)
    semaphore = asyncio.Semaphore(max_concurrent)

    for job in job_data:
        job_title = job.get("job_title", "Unknown")
        job_desc = job.get("job_description", "")
        job_context = f"{job_title} - {job_desc}"
        tasks = job.get("tasks", [])

        print(f"\n▶ Processing Occupation: {job_title} ({len(tasks)} tasks)")

        # Launch all tasks for this occupation concurrently
        coros = [
            evaluate_single_task(
                client, semaphore, completed, job_title, job_context, task, weights
            )
            for task in tasks
        ]
        occupation_results = await asyncio.gather(*coros)

        # Keep only successful evaluations
        valid = [r for r in occupation_results if r is not None]

        if not valid:
            print(f"  ⚠ No successful tasks for {job_title}")
            continue

        # Occupation-level weighted score
        total_weighted = sum(r["task_automation_score"] * r["importance"] for r in valid)
        total_importance = sum(r["importance"] for r in valid)
        occ_score = round(total_weighted / total_importance, 2) if total_importance else 0.0

        # Attach score and update in-memory cache
        for r in valid:
            r["occ_automation_score"] = occ_score
            completed[str(r["task_id"])] = r

        # Checkpoint after every occupation
        await save_checkpoint(checkpoint_path, list(completed.values()))
        print(f"  ✓ {job_title} finished — occ_score = {occ_score}")

    # Final save
    await save_checkpoint(checkpoint_path, list(completed.values()))
    return list(completed.values())

## Final Class

In [36]:
class AutomationAssessmentPipeline:
    def __init__(
        self,
        api_key: str,
        data_path: str,
        checkpoint_path: str = "automation_checkpoint.json",
        max_concurrent: int = 8,
    ):
        self.client = genai.Client(api_key=api_key)
        self.data_path = data_path
        self.checkpoint_path = checkpoint_path
        self.max_concurrent = max_concurrent
        self.job_data = load_job_data(data_path)

    async def run(self, weights: Dict[str, float]) -> List[Dict[str, Any]]:
        """Public entry point – processes every occupation and returns results."""
        return await process_all_occupations(
            client=self.client,
            job_data=self.job_data,
            weights=weights,
            checkpoint_path=self.checkpoint_path,
            max_concurrent=self.max_concurrent,
        )

In [23]:
'''
class AutomationAssessmentPipeline:
    def __init__(self, api_key: str, data_path: str):
        self.client = genai.Client(api_key=api_key)
        self.data_path = data_path
        self.job_data = self._load_json_data()

    def _load_json_data(self) -> List[Dict[str, Any]]:
        """Reads the prepared JSON dataset."""
        with open(self.data_path, 'r') as file:
            return json.load(file)

    def build_assessment_prompt(self, task_description: str, job_context: str) -> str:
        """Constructs prompt matching the exact framework prompt specification."""
        return f"""Analyze the following job task and evaluate its automation potential across 7 distinct dimensions.
For each dimension, assign a score from 1 to 5, where:
1 = Very difficult to automate (Strong human advantage / External barrier)
5 = Highly automatable (Strong AI/Machine advantage / No barrier)

Job Context: "{job_context}"
Task to Analyze: "{task_description}"

Dimensions to evaluate (Score 1-5):
1. Task Predictability: Is the internal logic heuristic/intuition-based (1) or strictly algorithmic and rule-based (5)?
2. Interaction Medium: Does the task require complex physical manipulation (1) or is it purely abstract digital processing (5)?
3. Social Requirement: Does the task demand deep empathy and trust-building (1) or can it be executed in total isolation (5)?
4. Environmental Stability: Does the work occur in a highly chaotic/unpredictable environment (1) or a perfectly engineered static environment (5)?
5. Consequence of Failure: Are the stakes of a mistake catastrophic (1) or trivial and easily reversed (5)?
6. Regulatory Barrier: Does the law strictly mandate a certified human (1) or is the output completely unregulated (5)?
7. Economic Arbitrage: Is human labor so cheap that automation ROI is negligible (1) or is human labor highly expensive making ROI massive (5)?

Provide your assessment STRICTLY in the following JSON format. Be specific and justify each score in the reasoning field BEFORE providing the score to ensure sound logic.

{{
  "task_predictability": {{"reasoning": "...", "score": X}},
  "interaction_medium": {{"reasoning": "...", "score": X}},
  "social_requirement": {{"reasoning": "...", "score": X}},
  "environmental_stability": {{"reasoning": "...", "score": X}},
  "consequence_of_failure": {{"reasoning": "...", "score": X}},
  "regulatory_barrier": {{"reasoning": "...", "score": X}},
  "economic_arbitrage": {{"reasoning": "...", "score": X}}
}}"""

    async def assess_task_automation(self, task_description: str, job_context: str) -> Dict[str, Any]:
        """Calls Gemini API via AsyncChat to structure response."""
        prompt = self.build_assessment_prompt(task_description, job_context)
        max_retries = 3
        backoff_factor = 15.0
        
        for attempt in range(max_retries):
            try:
                chat = self.client.aio.chats.create(
                    model='gemini-3.6-flash',
                    config=types.GenerateContentConfig(
                        temperature=0.2,
                        response_mime_type="application/json",
                        response_schema=TaskAssessment,
                        system_instruction="You are an expert in labor economics and AI automation."
                    )
                )
                
                response = await chat.send_message(prompt)
                return json.loads(response.text)
                
            except Exception as e:
                error_str = str(e)
                if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                    wait_time = backoff_factor * (attempt + 1)
                    print(f"Rate limit reached. Pausing for {wait_time}s (Attempt {attempt+1}/{max_retries})...")
                    await asyncio.sleep(wait_time)
                else:
                    print(f"Error evaluating task '{task_description[:30]}...': {e}")
                    break
        return {}

    def calculate_automation_score(self, dimension_scores: Dict[str, Any], weights: Dict[str, float]) -> float:
        """Calculates normalized overall automation potential score (0 to 100%)."""
        overall_score = 0.0
        for dim, details in dimension_scores.items():
            if dim in weights and isinstance(details, dict) and 'score' in details:
                normalized_score = (details['score'] / 5.0) * 100
                overall_score += normalized_score * weights[dim]
        return overall_score

    async def process_all_occupations(self, weights: Dict[str, float]) -> List[Dict[str, Any]]:
        """Processes occupations iteratively without artificial delays."""
        results = []
        for job in self.job_data:
            job_title = job.get('job_title', 'Unknown')
            job_desc = job.get('job_description', '')
            job_context = f"{job_title} - {job_desc}"
            
            print(f"\nProcessing Occupation: {job_title}")
            
            occupation_task_results = []
            tasks = job.get('tasks', [])
            total_tasks = len(tasks)
            
            for idx, task in enumerate(tasks, start=1):
                task_desc = task['task_description']
                importance = task.get('importance')
                
                # Assign default weight of 1.0 for any tasks with missing or null importance data
                importance_val = float(importance) if importance is not None else 1.0
                
                print(f"  -> Evaluating task ({idx}/{total_tasks}): {task_desc[:40]}...")
                assessment = await self.assess_task_automation(task_desc, job_context)
                
                if assessment:
                    task_score = self.calculate_automation_score(assessment, weights)
                    occupation_task_results.append({
                        "job_title": job_title,
                        "task_id": task['task_id'],
                        "task_description": task_desc,
                        "dimension_assessments": assessment,
                        "task_automation_score": task_score,
                        "importance": importance_val
                    })
                
            # Calculate the weighted Occupation Automation Score
            if occupation_task_results:
                total_weighted_score = sum(res['task_automation_score'] * res['importance'] for res in occupation_task_results)
                total_importance = sum(res['importance'] for res in occupation_task_results)
                occ_automation_score = total_weighted_score / total_importance if total_importance > 0 else 0.0
                
                # Append final structured layout matching the desired df_results output
                for res in occupation_task_results:
                    results.append({
                        "job_title": res["job_title"],
                        "task_id": res["task_id"],
                        "task_description": res["task_description"],
                        "dimension_assessments": res["dimension_assessments"],
                        "task_automation_score": res["task_automation_score"],
                        "occ_automation_score": round(occ_automation_score, 2)
                    })
                
        return results
'''

'\nclass AutomationAssessmentPipeline:\n    def __init__(self, api_key: str, data_path: str):\n        self.client = genai.Client(api_key=api_key)\n        self.data_path = data_path\n        self.job_data = self._load_json_data()\n\n    def _load_json_data(self) -> List[Dict[str, Any]]:\n        """Reads the prepared JSON dataset."""\n        with open(self.data_path, \'r\') as file:\n            return json.load(file)\n\n    def build_assessment_prompt(self, task_description: str, job_context: str) -> str:\n        """Constructs prompt matching the exact framework prompt specification."""\n        return f"""Analyze the following job task and evaluate its automation potential across 7 distinct dimensions.\nFor each dimension, assign a score from 1 to 5, where:\n1 = Very difficult to automate (Strong human advantage / External barrier)\n5 = Highly automatable (Strong AI/Machine advantage / No barrier)\n\nJob Context: "{job_context}"\nTask to Analyze: "{task_description}"\n\nDimensions

# Weights

In [24]:
# Exact weight distribution from framework specification
dimension_weights = {
    "task_predictability": 0.15,
    "interaction_medium": 0.15,
    "social_requirement": 0.15,
    "environmental_stability": 0.15,
    "consequence_of_failure": 0.15,
    "regulatory_barrier": 0.15,
    "economic_arbitrage": 0.10
}

# Initialize Pipeline

In [32]:
# Load API Key from local text file
with open("../api_key.txt", "r") as f:
    API_KEY = f.read().strip()

DATA_PATH = "../data/processed/unified_data.json"
CHECKPOINT_PATH = "../data/processed/automation_checkpoint.json"

pipeline = AutomationAssessmentPipeline(
    api_key=API_KEY,
    data_path=DATA_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    max_concurrent=6,
)

# Test Variance

In [33]:
target_job = "Telemarketers"
iterations = 5

original_job_data = pipeline.job_data
filtered_jobs = [job for job in original_job_data if job.get("job_title") == target_job]

if not filtered_jobs:
    print(f"Occupation '{target_job}' not found in dataset.")
else:
    all_results = []

    for i in range(iterations):
        print(f"\nRunning iteration {i+1}/{iterations}...")

        # Create a unique path that does NOT yet exist as a file
        temp_checkpoint = Path(tempfile.gettempdir()) / f"variance_ckpt_{i}_{os.getpid()}.json"

        run_assessments = await process_all_occupations(
            client=pipeline.client,
            job_data=filtered_jobs,
            weights=dimension_weights,
            checkpoint_path=temp_checkpoint,
            max_concurrent=pipeline.max_concurrent,
        )

        # Clean up
        temp_checkpoint.unlink(missing_ok=True)

        if run_assessments:
            for task_result in run_assessments:
                all_results.append({
                    "iteration": i + 1,
                    "job_title": target_job,
                    "task_id": task_result["task_id"],
                    "task_description": task_result["task_description"],
                    "task_automation_score": task_result["task_automation_score"],
                    "occ_automation_score": task_result["occ_automation_score"],
                })

    # ---------- reports (unchanged) ----------
    df_variance = pd.DataFrame(all_results)

    occ_constancy_report = (
        df_variance.groupby("job_title")["occ_automation_score"]
        .agg(["min", "max", "mean", "std"])
        .round(2)
        .reset_index()
    )

    task_constancy_report = (
        df_variance.groupby(["task_id", "task_description"])["task_automation_score"]
        .agg(["min", "max", "mean", "std"])
        .round(2)
        .reset_index()
    )

    print("\n--- Occupation Automation Score Variance Report ---")
    display(occ_constancy_report)

    print("\n--- Task Automation Score Variance Report ---")
    display(task_constancy_report)


Running iteration 1/5...

▶ Processing Occupation: Telemarketers (12 tasks)
  → Telemarketers | task 4619: Contact businesses or private individuals by telephone …
  → Telemarketers | task 4621: Obtain customer information such as name, address, and …
  → Telemarketers | task 4620: Explain products or services and prices, and answer que…
  → Telemarketers | task 4622: Record names, addresses, purchases, and reactions of pr…
  → Telemarketers | task 4627: Maintain records of contacts, accounts, and orders.…
  → Telemarketers | task 4625: Answer telephone calls from potential customers who hav…
  → Telemarketers | task 4618: Deliver prepared sales talks, reading from scripts that…
  → Telemarketers | task 4626: Telephone or write letters to respond to correspondence…
  → Telemarketers | task 4623: Adjust sales scripts to better target the needs and int…
  → Telemarketers | task 4624: Obtain names and telephone numbers of potential custome…
  → Telemarketers | task 4628: Schedule appoint

,job_title,min,max,mean,std
0,Telemarketers,86.02,88.41,87.09,0.98



--- Task Automation Score Variance Report ---


,task_id,task_description,min,max,mean,std
0,4618,"Deliver prepared sales talks, reading from scr...",80.0,86.0,83.6,2.51
1,4619,Contact businesses or private individuals by t...,77.0,86.0,83.0,3.67
2,4620,"Explain products or services and prices, and a...",81.0,83.0,82.6,0.89
3,4621,"Obtain customer information such as name, addr...",89.0,92.0,90.8,1.64
4,4622,"Record names, addresses, purchases, and reacti...",90.0,95.0,93.4,2.30
5,4623,Adjust sales scripts to better target the need...,86.0,92.0,89.0,3.00
6,4624,Obtain names and telephone numbers of potentia...,95.0,98.0,96.8,1.64
7,4625,Answer telephone calls from potential customer...,81.0,83.0,82.6,0.89
8,4626,Telephone or write letters to respond to corre...,81.0,83.0,81.4,0.89
9,4627,"Maintain records of contacts, accounts, and or...",95.0,95.0,95.0,0.00


# All Occupations

In [37]:
final_assessments = await pipeline.run(dimension_weights)


▶ Processing Occupation: Software Developers (17 tasks)
  ✓ Software Developers finished — occ_score = 76.89

▶ Processing Occupation: Data Scientists (54 tasks)
  ✓ Data Scientists finished — occ_score = 80.56

▶ Processing Occupation: Registered Nurses (137 tasks)
  ✓ Registered Nurses finished — occ_score = 48.75

▶ Processing Occupation: Radiologic Technologists (30 tasks)
  ✓ Radiologic Technologists finished — occ_score = 58.1

▶ Processing Occupation: Pharmacists (20 tasks)
  ✓ Pharmacists finished — occ_score = 64.18

▶ Processing Occupation: Elementary School Teachers (38 tasks)
  ✓ Elementary School Teachers finished — occ_score = 50.45

▶ Processing Occupation: Lawyers (22 tasks)
  ✓ Lawyers finished — occ_score = 55.93

▶ Processing Occupation: Paralegals and Legal Assistants (12 tasks)
  ✓ Paralegals and Legal Assistants finished — occ_score = 68.62

▶ Processing Occupation: Financial Analysts (26 tasks)
  ✓ Financial Analysts finished — occ_score = 68.15

▶ Processing Oc

# Results

In [ ]:
df_results = pd.DataFrame(final_assessments)
df_results.head()

,job_title,task_id,task_description,dimension_assessments,task_automation_score,importance,occ_automation_score
0,Software Developers,21662,Analyze user needs and software requirements t...,{'task_predictability': {'reasoning': 'Determi...,70.0,4.08,76.89
1,Software Developers,21669,Develop or direct software system testing or v...,{'task_predictability': {'reasoning': 'Develop...,82.0,3.90,76.89
2,Software Developers,21664,"Confer with systems analysts, engineers, progr...",{'task_predictability': {'reasoning': 'Conferr...,70.0,3.75,76.89
3,Software Developers,21670,"Modify existing software to correct errors, ad...",{'task_predictability': {'reasoning': 'Modifyi...,85.0,3.74,76.89
4,Software Developers,21673,Prepare reports or correspondence concerning p...,{'task_predictability': {'reasoning': 'Prepari...,91.0,3.68,76.89


Drop Janitors and Cleaners occupation, all task were not analyzed

Pipeline runtime needs to be reduced

In [ ]:
file_path = '../src/automation_assessments.json'

# Read the JSON file
with open(file_path, 'r') as file:
    job_data = json.load(file)

# Filter out "Janitors and Cleaners"
filtered_data = [job for job in job_data if job.get("job_title") != "Janitors and Cleaners"]

# Save the updated data back to the file
with open(file_path, 'w') as file:
    json.dump(filtered_data, file, indent=4)